# CatVTON — LoRA Fine-Tuning on RunPod
### `train_lora_modified.py` — full control over every hyperparam

**Run cells top-to-bottom once per pod session.**  
Only §1 (install) and §2 (clone) need to run once; everything else is re-runnable.

---
## §1 — Install dependencies
> RunPod already has CUDA-enabled PyTorch. We skip reinstalling torch/torchvision to avoid
> replacing the GPU build with a CPU-only one from PyPI.

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f'Command failed: {cmd}')
    print(result.stdout[-2000:] or '(done)')

# Verify GPU is available before wasting time installing
import torch
assert torch.cuda.is_available(), 'No GPU detected — check RunPod pod type!'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}  |  PyTorch: {torch.__version__}')

In [ ]:
run('pip install -q --upgrade pip')

# Pin versions that are known to work together
packages = [
    'diffusers==0.29.2',
    'transformers==4.46.3',
    'accelerate==0.33.0',
    'peft==0.12.0',          # >=0.12 has get_peft_model; pinned for reproducibility
    'huggingface_hub>=0.24.0,<2.0',
    'safetensors',
    'numpy==1.26.4',
    'pillow==10.3.0',
    'opencv-python-headless==4.10.0.84',  # headless: no GUI deps on server
    'scipy==1.13.1',
    'scikit-image==0.24.0',
    'fvcore==0.1.5.post20221221',
    'cloudpickle==3.0.0',
    'omegaconf==2.3.0',
    'pycocotools==2.0.8',
    'PyYAML==6.0.1',
    'tqdm==4.66.4',
    'matplotlib==3.9.1',
]

run('pip install -q ' + ' '.join(f'"{p}"' for p in packages))
print('All packages installed.')

---
## §2 — Clone repo & set working directory

In [ ]:
import os

REPO_URL    = 'https://github.com/Ahmed3280/LoRA_Fashion.git'
BRANCH      = 'main'
REPO_DIR    = '/workspace/LoRA_Fashion'

if not os.path.exists(REPO_DIR):
    run(f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
    print('Repo cloned.')
else:
    run(f'git -C {REPO_DIR} pull origin {BRANCH}')
    print('Repo updated.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

---
## §3 — Mount / upload dataset

Set `DATA_ROOT` to wherever your dataset lives inside the pod.  
Common options:
- **RunPod Network Volume** (recommended for large datasets): `/runpod-volume/mena_dataset`
- **Uploaded manually** via the RunPod file browser: `/workspace/mena_dataset`

Expected layout:
```
DATA_ROOT/
  train/
    image/           ← person JPGs
    cloth/           ← garment JPGs
    agnostic-mask/   ← PNG masks (same stem as image/)
    train_pairs.txt  ← "person.jpg cloth.jpg" per line
```

In [ ]:
from pathlib import Path

DATA_ROOT = '/workspace/mena_dataset'   # <-- change this if needed

# Quick sanity check
required = [
    Path(DATA_ROOT) / 'train' / 'image',
    Path(DATA_ROOT) / 'train' / 'cloth',
    Path(DATA_ROOT) / 'train' / 'agnostic-mask',
    Path(DATA_ROOT) / 'train' / 'train_pairs.txt',
]
for p in required:
    status = '✓' if p.exists() else '✗  MISSING'
    print(f'  {status}  {p}')

pairs = (Path(DATA_ROOT) / 'train' / 'train_pairs.txt').read_text().strip().splitlines()
print(f'\nTotal pairs in train_pairs.txt: {len(pairs)}')

---
## §4 — Hyperparameters
**Edit this cell only.** Everything below uses these variables.

In [ ]:
# ── Model checkpoints ────────────────────────────────────────────────────────
BASE_CKPT           = 'booksforcharlie/stable-diffusion-inpainting'
ATTN_CKPT           = 'zhengchong/CatVTON'
ATTN_CKPT_VERSION   = 'mix'          # 'mix' | 'vitonhd' | 'dresscode'

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_DIR          = '/workspace/output/lora_mena'

# ── Image resolution ─────────────────────────────────────────────────────────
HEIGHT              = 512            # 512 for low VRAM (16 GB), 1024 for 40+ GB
WIDTH               = 384

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE          = 1
GRAD_ACCUM_STEPS    = 4              # effective batch = BATCH_SIZE × GRAD_ACCUM_STEPS
NUM_EPOCHS          = 20
MIXED_PRECISION     = 'bf16'         # 'no' | 'fp16' | 'bf16'
GRADIENT_CKPT       = True           # True = saves VRAM (~40%); False = faster

# ── Optimiser ────────────────────────────────────────────────────────────────
LR                  = 1e-4
LR_SCHEDULER        = 'cosine'       # 'constant' | 'cosine' | 'cosine_with_restarts' | 'linear' | 'polynomial'
LR_WARMUP_STEPS     = 100

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_RANK           = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = 'to_q,to_k,to_v,to_out.0'   # comma-separated

# ── Validation / early stopping ───────────────────────────────────────────────
VAL_SPLIT           = 0.1           # fraction of pairs held out for validation (0 = disabled)
VAL_SEED            = 42
EVAL_EVERY          = 1             # run val every N epochs
EARLY_STOP_PATIENCE = 5             # stop if no improvement for N evals (0 = disabled)
EARLY_STOP_MIN_DELTA= 0.0

# ── Misc ─────────────────────────────────────────────────────────────────────
CLOTH_TYPE          = 'overall'      # 'upper' | 'lower' | 'overall' | 'inner' | 'outer'
SAVE_EVERY          = 5             # save checkpoint every N epochs
NUM_WORKERS         = 2
SEED                = 42

print('Hyperparameters set.')
print(f'  Effective batch size : {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print(f'  Resolution           : {WIDTH}x{HEIGHT}')
print(f'  LoRA rank/alpha      : {LORA_RANK}/{LORA_ALPHA}')
print(f'  Epochs               : {NUM_EPOCHS}')
print(f'  LR / scheduler       : {LR} / {LR_SCHEDULER}')
print(f'  Output               : {OUTPUT_DIR}')

---
## §5 — Configure Accelerate (single-GPU, no prompt)
Writes `~/.cache/huggingface/accelerate/default_config.yaml` so
`accelerate launch` works without an interactive wizard.

In [ ]:
import yaml, os

acc_cfg = {
    'compute_environment': 'LOCAL_MACHINE',
    'debug': False,
    'distributed_type': 'NO',
    'downcast_bf16': 'no',
    'gpu_ids': 'all',
    'machine_rank': 0,
    'main_training_function': 'main',
    'mixed_precision': MIXED_PRECISION,
    'num_machines': 1,
    'num_processes': 1,
    'rdzv_backend': 'static',
    'same_network': True,
    'tpu_env': [],
    'tpu_use_cluster': False,
    'tpu_use_sudo': False,
    'use_cpu': False,
}

cfg_dir = os.path.expanduser('~/.cache/huggingface/accelerate')
os.makedirs(cfg_dir, exist_ok=True)
with open(os.path.join(cfg_dir, 'default_config.yaml'), 'w') as f:
    yaml.dump(acc_cfg, f)

print('Accelerate config written.')

---
## §6 — Pre-download model weights
Downloads to HuggingFace cache so training doesn't time-out mid-run.

In [ ]:
import os
os.environ['HF_HUB_HTTP_TIMEOUT'] = '120'

from huggingface_hub import snapshot_download

print('Downloading base SD-inpainting model...')
snapshot_download(repo_id=BASE_CKPT)
print('Done.')

print('Downloading CatVTON attention checkpoint...')
snapshot_download(repo_id=ATTN_CKPT)
print('Done.')

print('Downloading VAE...')
snapshot_download(repo_id='stabilityai/sd-vae-ft-mse')
print('All weights cached.')

---
## §7 — VRAM estimate
Quick check before committing to a long run.

In [ ]:
import torch
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'Total VRAM : {total_vram:.1f} GB')

# rough estimates for 512x384 vs 1024x768
if HEIGHT <= 512:
    est = 10
    label = '512×384 (low-res)'
else:
    est = 22
    label = '1024×768 (full-res)'

print(f'Estimated peak VRAM for {label} + bf16 + grad-ckpt: ~{est} GB')
if total_vram < est:
    print(f'WARNING: may OOM — lower HEIGHT/WIDTH or enable gradient checkpointing')
else:
    print('Should fit comfortably.')

---
## §8 — Build training command & launch
Assembles the `accelerate launch` command from your §4 hyperparameters and runs it.
Output streams live to the cell below.

In [ ]:
import shlex

no_grad_ckpt_flag = '' if GRADIENT_CKPT else '--no_gradient_checkpointing'

cmd_parts = [
    'accelerate launch',
    'train_lora_modified.py',
    f'--data_root            "{DATA_ROOT}"',
    f'--base_ckpt            "{BASE_CKPT}"',
    f'--attn_ckpt            "{ATTN_CKPT}"',
    f'--attn_ckpt_version    {ATTN_CKPT_VERSION}',
    f'--output_dir           "{OUTPUT_DIR}"',
    f'--cloth_type           {CLOTH_TYPE}',
    f'--height               {HEIGHT}',
    f'--width                {WIDTH}',
    f'--batch_size           {BATCH_SIZE}',
    f'--gradient_accumulation_steps {GRAD_ACCUM_STEPS}',
    f'--num_epochs           {NUM_EPOCHS}',
    f'--mixed_precision      {MIXED_PRECISION}',
    f'--lr                   {LR}',
    f'--lr_scheduler         {LR_SCHEDULER}',
    f'--lr_warmup_steps      {LR_WARMUP_STEPS}',
    f'--lora_rank            {LORA_RANK}',
    f'--lora_alpha           {LORA_ALPHA}',
    f'--lora_dropout         {LORA_DROPOUT}',
    f'--lora_target_modules  "{LORA_TARGET_MODULES}"',
    f'--val_split            {VAL_SPLIT}',
    f'--val_seed             {VAL_SEED}',
    f'--eval_every           {EVAL_EVERY}',
    f'--early_stop_patience  {EARLY_STOP_PATIENCE}',
    f'--early_stop_min_delta {EARLY_STOP_MIN_DELTA}',
    f'--save_every           {SAVE_EVERY}',
    f'--num_workers          {NUM_WORKERS}',
    f'--seed                 {SEED}',
    no_grad_ckpt_flag,
]

TRAIN_CMD = ' '.join(p for p in cmd_parts if p.strip())
print('Training command:')
print(TRAIN_CMD)

In [ ]:
import subprocess, sys, os

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ['HF_HUB_HTTP_TIMEOUT'] = '120'

# Stream output live
process = subprocess.Popen(
    TRAIN_CMD,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0:
    print('\n✓ Training finished successfully.')
else:
    print(f'\n✗ Training exited with code {process.returncode}')

---
## §9 — Inspect saved checkpoints

In [ ]:
from pathlib import Path

out = Path(OUTPUT_DIR)
if not out.exists():
    print('Output directory not found — training may not have started.')
else:
    checkpoints = sorted(out.iterdir())
    print(f'Checkpoints in {OUTPUT_DIR}:')
    for ckpt in checkpoints:
        files = list(ckpt.iterdir()) if ckpt.is_dir() else []
        file_names = ', '.join(f.name for f in files)
        print(f'  {ckpt.name}/  →  {file_names}')

---
## §10 — (Optional) Copy final weights to Network Volume
Keeps the LoRA weights safe if the pod is terminated.

In [ ]:
import shutil
from pathlib import Path

NETWORK_VOLUME = '/runpod-volume'   # change if your volume is mounted elsewhere
SAVE_TO = f'{NETWORK_VOLUME}/lora_final'

src = Path(OUTPUT_DIR) / 'final'
if src.exists():
    shutil.copytree(src, SAVE_TO, dirs_exist_ok=True)
    print(f'Copied {src}  →  {SAVE_TO}')
else:
    print(f'No final/ checkpoint found yet at {src}')